# L1 and L2 Regularization

## Theory

### What is Regularization?
Regularization is a technique used to prevent overfitting by adding a penalty term to the loss function. The two most common types are L1 (Lasso) and L2 (Ridge) regularization.

### L1 Regularization (Lasso)
- Adds the absolute value of coefficients as penalty term: `λ * Σ|w|`
- Tends to produce sparse models (feature selection)
- Some coefficients become exactly zero
- Good for feature selection

### L2 Regularization (Ridge)
- Adds squared magnitude of coefficients as penalty term: `λ * Σw²`
- Shrinks coefficients towards zero but not exactly zero
- Better for handling multicollinearity
- More stable solutions

### Elastic Net
- Combines both L1 and L2 regularization
- Penalty term: `λ * (α|w| + (1-α)w²)`
- Provides benefits of both approaches

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Generate synthetic data with multicollinearity
np.random.seed(42)
n_samples = 100
n_features = 20

# Create correlated features
X = np.random.randn(n_samples, n_features)
X[:, 5:] = X[:, :15] + np.random.randn(n_samples, 15) * 0.1

# True coefficients: only first 5 features are relevant
true_coef = np.zeros(n_features)
true_coef[:5] = [2.0, -1.0, 1.5, -0.5, 1.0]

# Generate target variable with noise
y = np.dot(X, true_coef) + np.random.randn(n_samples) * 0.1

# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train different models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=1.0),
    'Lasso (L1)': Lasso(alpha=1.0),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5)
}

# Fit models and store coefficients
coefficients = {}
scores = {}

for name, model in models.items():
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Store coefficients
    coefficients[name] = model.coef_
    
    # Calculate scores
    y_pred = model.predict(X_test_scaled)
    scores[name] = {
        'R2': r2_score(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred)
    }

# Plot coefficients
plt.figure(figsize=(15, 6))
x = np.arange(n_features)

for name, coef in coefficients.items():
    plt.plot(x, coef, 'o-', label=name, alpha=0.7)

plt.plot(x, true_coef, 'k--', label='True coefficients', alpha=0.5)
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
plt.xlabel('Feature index')
plt.ylabel('Coefficient value')
plt.title('Comparison of Regularization Methods')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Print scores
print("\nModel Performance:")
for name, metrics in scores.items():
    print(f"\n{name}:")
    print(f"R² Score: {metrics['R2']:.4f}")
    print(f"MSE: {metrics['MSE']:.4f}")

## Effect of Regularization Strength (α)

Let's see how different values of the regularization parameter α affect the coefficients.

In [ ]:
# Test different alpha values
alphas = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]

# Store coefficients for different alphas
ridge_coefs = []
lasso_coefs = []

for alpha in alphas:
    # Ridge
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    ridge_coefs.append(ridge.coef_)
    
    # Lasso
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train_scaled, y_train)
    lasso_coefs.append(lasso.coef_)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Ridge plot
ax1.set_prop_cycle(color=[plt.cm.viridis(i) for i in np.linspace(0, 1, len(alphas))])
for alpha, coef in zip(alphas, ridge_coefs):
    ax1.plot(x, coef, 'o-', label=f'α = {alpha}', alpha=0.7)
ax1.plot(x, true_coef, 'k--', label='True', alpha=0.5)
ax1.set_title('Ridge Regression (L2)')
ax1.set_xlabel('Feature index')
ax1.set_ylabel('Coefficient value')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Lasso plot
ax2.set_prop_cycle(color=[plt.cm.viridis(i) for i in np.linspace(0, 1, len(alphas))])
for alpha, coef in zip(alphas, lasso_coefs):
    ax2.plot(x, coef, 'o-', label=f'α = {alpha}', alpha=0.7)
ax2.plot(x, true_coef, 'k--', label='True', alpha=0.5)
ax2.set_title('Lasso Regression (L1)')
ax2.set_xlabel('Feature index')
ax2.set_ylabel('Coefficient value')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## Key Observations

### L1 (Lasso) Regularization
1. Performs feature selection by setting some coefficients exactly to zero
2. More aggressive at eliminating less important features
3. Solution can be unstable when features are highly correlated

### L2 (Ridge) Regularization
1. Shrinks all coefficients proportionally
2. Never sets coefficients exactly to zero
3. Better handles multicollinearity
4. More stable solutions when features are correlated

### Elastic Net
1. Combines benefits of both L1 and L2
2. Good choice when you want both feature selection and handling of correlated features

## When to Use What?

### Use L1 (Lasso) when:
- You want automatic feature selection
- You believe only a few features are important
- Features are not highly correlated

### Use L2 (Ridge) when:
- You have multicollinearity
- You want to keep all features but with reduced impact
- You want more stable solutions

### Use Elastic Net when:
- You want a balance between L1 and L2
- You're not sure which regularization to use
- You have both correlated features and need feature selection